<a href="https://colab.research.google.com/github/adzetto/marine_analysis/blob/main/su-seviyesi/colab_baslat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bozyazı deniz seviyesi analizi — Colab

Hocanın istediği çıktılar:

1. Ayıklanmış veriden **MSL**
2. **Gelgit bileşenleri** tablosu — ad, genlik [m], faz [derece]
   (Famagusta Tablo 2-2 biçiminde)
3. **Gelgit düzeyleri** tablosu — HAT, MHWS, MHHW, MHW, MHWN, MSL,
   MLWN, MLW, MLLW, MLWS, LAT [m +MSL] (Tablo 2-3 biçiminde)
4. **Gelgit dışı** su yükselmeleri — ortalama, dağılım ve PDF/CDF grafiği

Analiz dönemi makalenin Bozyazı penceresiyle aynı: **01.01.2010 –
13.03.2018**. (Makale 2009'da başlıyor ama portalda Bozyazı kaydı 2010
başında başlıyor; bitiş tarihi birebir aynı tutuldu.)

**Çalışma zamanını yüksek RAM'e alın.** 15 dakikalık çözünürlükte
312.000 nokta için UTide'ın tasarım matrisi birkaç GB istiyor.

`Çalışma zamanı → Çalışma zamanı türünü değiştir → Yüksek RAM`

## 1. Kurulum

Bu hücre tekrar tekrar çalıştırılabilir; her seferinde temiz klon alır.
(Önceki sürümde art arda çalıştırınca dizin kendi içine klonlanıp
çıktılar iç içe klasörlere dağılmıştı.)

In [ ]:
%cd /content
!rm -rf /content/marine_analysis
!git clone -q https://github.com/adzetto/marine_analysis.git
%cd /content/marine_analysis/su-seviyesi
!pip install -q -r requirements.txt
!pwd

LaTeX isteğe bağlı — kurulmazsa figürler matplotlib'in kendi matematik
dizgisiyle üretilir, betikler bunu kendisi algılar.

In [ ]:
!apt-get -qq update && apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super > /dev/null
print('latex kuruldu')

In [ ]:
import psutil, os
gb = psutil.virtual_memory().total / 1e9
print(f'toplam RAM : {gb:.1f} GB   CPU: {os.cpu_count()} cekirdek')
if gb < 20:
    print('UYARI: yuksek RAM calisma zamanina gecmeniz onerilir.')

## 2. Veri

Veri depoyla geliyor. `tudes.harita.gov.tr` Türkiye dışındaki DNS
çözümleyicilerinden çözülmediği için indirme Colab'da çalışmaz.

In [ ]:
from ortak import oku, kes, PAPER_BAS, PAPER_BIT
s = oku()
x = kes(s, PAPER_BAS, PAPER_BIT)
print(f'tum ayiklanmis seri : {len(s):,} kayit  '
      f'({s.index.min().date()} -> {s.index.max().date()})')
print(f'analiz penceresi    : {len(x):,} kayit  '
      f'({PAPER_BAS} -> {PAPER_BIT})')
print(f'MSL (bu pencerede)  : {x.mean():.4f} m  '
      f'[istasyonun yerel datumu]')

## 3. Ayıklama

Depodaki `bozyazi_temiz.dat.gz` zaten bu adımın çıktısı. Yeniden üretmek
isterseniz çalıştırın; `04` ayıklamanın doğru şeyi sildiğini sınar.

In [ ]:
!python -u 03_veri_ayikla.py
!python -u 04_ayiklama_dogrula.py

## 4. Gelgit bileşenleri  (hocanın 2. isteği)

UTide harmonik çözümü + yayımlanmış Bozyazı değerleriyle karşılaştırma.
Ağır adım bu.

In [ ]:
!python -u 05_harmonik_analiz.py

## 5. Gelgit düzeyleri  (hocanın 3. isteği)

In [ ]:
!python -u 06_gelgit_seviyeleri.py

## 6. Gelgit dışı su yükselmeleri  (hocanın 4. isteği)

Ortalama, standart sapma, yüzdelikler ve PDF/CDF grafiği.

In [ ]:
!python -u 07_non_tidal.py

## 7. Deniz seviyesi değişimi  (hocanın 1. cümlesi)

Bozyazı'daki yıllar arası değişimin bölgesel mi istasyona özgü mü
olduğunu komşu istasyonlarla (Taşucu, Erdemli, Antalya) karşılaştırarak
sınar.

Komşu istasyonların ham verisi depoda değil (her biri ~33 MB), bu yüzden
Colab'da bu betik atlanır. Sonucu `tables/06_istasyon_yillik_sapma.csv`
ve `figures/03_istasyon_karsilastirma.png` olarak depoda hazır.

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('tables/06_istasyon_yillik_sapma.csv'))
display(Image('figures/03_istasyon_karsilastirma.png'))

## 8. Sonuçları göster

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image

for f in sorted(glob.glob('tables/*.csv')):
    print('=' * 70); print(f); print('=' * 70)
    display(pd.read_csv(f))

for f in sorted(glob.glob('figures/*.png')):
    print(f)
    display(Image(f))

## 9. Çıktıları indir

In [ ]:
%cd /content/marine_analysis/su-seviyesi
!zip -qr /content/bozyazi_sonuclar.zip tables figures data
from google.colab import files
files.download('/content/bozyazi_sonuclar.zip')